# Supervised Models Notebook

This notebook is for the supervised part of Week 1.
It builds features from the VynFi journal entries, then trains:

- logistic regression
- histogram gradient boosting

The notebook keeps the workflow clear and easy to follow.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler


In [2]:
from pathlib import Path

# This makes the notebook work whether you open it from the repo root
# or from inside the notebooks folder.
if (Path.cwd() / "data").exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    REPO_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Could not find the repo root")

DATA_DIR = REPO_ROOT / "data" / "training" / "vynfi"
OUT_DIR = REPO_ROOT / "data" / "generated"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Data folder:", DATA_DIR)


Repo root: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint
Data folder: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint\data\training\vynfi


In [3]:
# The training data comes in three parts, so I read all of them before modelling.
shard_names = [
    "train-00000-of-00003.parquet",
    "train-00001-of-00003.parquet",
    "train-00002-of-00003.parquet",
]

frames = [pd.read_parquet(DATA_DIR / name) for name in shard_names]
main_data = pd.concat(frames, ignore_index=True)
print("Full data shape:", main_data.shape)
main_data.head()


Full data shape: (667584, 48)


,document_id,company_code,fiscal_year,fiscal_period,posting_date,document_date,document_type,currency,exchange_rate,reference,...,tax_code,transaction_id,account_class,account_class_name,account_sub_class,account_sub_class_name,predecessor_line_id,trading_partner,fraud_type,anomaly_type
0,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,d35dc4bc-97c1-5ed5-943c-c309f2fc0910,A.A,Cash & Cash Equivalents,A.A.A,Operating Cash,None,None,None,LatePosting
1,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,35c015fd-d1ff-5249-b762-f358654e2380,A.D,Prepaid Expenses & Other Current Assets,A.D.A,Prepaid Expenses,None,None,None,LatePosting
2,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,dad767d0-5c32-5eb4-9307-15d880670d07,A.H,Other Long-term Assets,A.H.A,Other Assets,None,None,None,LatePosting
3,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,a45c4cbd-98a0-5303-9a2e-752cd56bebb1,A.B,Trade Receivables,A.B.A,Trade Accounts Receivable,None,None,None,LatePosting
4,019e9be6-cfc8-76f0-a5b0-b64ba37df4bf,1000,2024,1,2024-01-09,2024-01-01,OPENING_BALANCE,USD,1.0,None,...,NaN,b1b232bc-c7bf-5d91-9ead-714e740ec1eb,A.C,Inventory,A.C.A,Inventory,None,None,None,LatePosting


In [4]:
empty_columns = [
    "auxiliary_account_number",
    "auxiliary_account_label",
    "lettrage",
    "lettrage_date",
    "tax_code",
]
leakage_columns = ["fraud_type", "anomaly_type", "is_anomaly"]

# Empty columns do not add useful information, so I remove them first.
work_df = main_data.drop(columns=empty_columns, errors="ignore")

# Grouping by document_id keeps every line of one journal entry in the same split.
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_index, test_index = next(
    splitter.split(work_df, y=work_df["is_fraud"], groups=work_df["document_id"])
)

train_df = work_df.iloc[train_index].reset_index(drop=True)
test_df = work_df.iloc[test_index].reset_index(drop=True)

# Keep fraud_type only for the final analysis table.
eval_fraud_type = test_df["fraud_type"].copy()

y_train = train_df["is_fraud"].astype(int)
y_test = test_df["is_fraud"].astype(int)

train_df = train_df.drop(columns=leakage_columns, errors="ignore")
test_df = test_df.drop(columns=leakage_columns, errors="ignore")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train fraud rate:", y_train.mean())
print("Test fraud rate:", y_test.mean())


Train shape: (536395, 40)
Test shape: (131189, 40)
Train fraud rate: 0.062409232002535446
Test fraud rate: 0.06194879143830657


In [5]:
numeric_features = [
    "amount",
    "log10_amount",
    "first_digit",
    "posting_lag_days",
    "posting_dayofweek",
    "exchange_rate",
    "line_number",
    "fiscal_period",
]

boolean_features = [
    "is_debit",
    "is_round_100",
    "is_round_1000",
    "is_weekend",
    "is_manual",
    "is_post_close",
]

categorical_features = [
    "document_type",
    "currency",
    "business_process",
    "business_unit",
    "account_class",
    "account_sub_class",
    "financial_statement_category",
    "source_system",
    "company_code",
]

account_feature = "gl_account"


# I create a smaller set of features that have a possible accounting meaning.
def build_features(df):
    out = pd.DataFrame(index=df.index)

    debit = df["debit_amount"].fillna(0)
    credit = df["credit_amount"].fillna(0)
    amount = (debit + credit).abs()

    # The log amount reduces the effect of a few extremely large transactions.
    out["amount"] = amount
    out["log10_amount"] = np.log10(amount.clip(lower=0.01))

    scaled = amount.clip(lower=0.01) / np.power(
        10.0,
        np.floor(np.log10(amount.clip(lower=0.01))),
    )
    out["first_digit"] = scaled.astype(int).clip(1, 9)

    # Round and debit indicators may help identify unusual manual-looking entries.
    out["is_debit"] = (debit > 0).astype(int)
    out["is_round_100"] = ((amount > 0) & (amount % 100 == 0)).astype(int)
    out["is_round_1000"] = ((amount > 0) & (amount % 1000 == 0)).astype(int)

    # Date features can show delayed postings or entries made during weekends.
    lag = (df["posting_date"] - df["document_date"]).dt.days
    out["posting_lag_days"] = lag.fillna(0)
    out["posting_dayofweek"] = df["posting_date"].dt.dayofweek.fillna(0)
    out["is_weekend"] = (df["posting_date"].dt.dayofweek >= 5).astype(int)

    out["exchange_rate"] = df["exchange_rate"].fillna(1.0)
    out["line_number"] = df["line_number"].fillna(0)
    out["fiscal_period"] = df["fiscal_period"].fillna(0)
    out["is_manual"] = df["is_manual"].astype(int)
    out["is_post_close"] = df["is_post_close"].astype(int)

    # Missing text values get their own label instead of being silently dropped.
    for column in categorical_features:
        out[column] = df[column].astype("string").fillna("missing")

    out[account_feature] = df[account_feature].fillna(-1).astype("int64")
    return out


X_train = build_features(train_df)
X_test = build_features(test_df)
print("Train feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)
X_train.head()


Train feature shape: (536395, 24)
Test feature shape: (131189, 24)


,amount,log10_amount,first_digit,is_debit,is_round_100,is_round_1000,posting_lag_days,posting_dayofweek,is_weekend,exchange_rate,...,document_type,currency,business_process,business_unit,account_class,account_sub_class,financial_statement_category,source_system,company_code,gl_account
0,5.620588e+05,5.749782,5,1,0,0,8,1,0,1.0,...,OPENING_BALANCE,USD,R2R,missing,A.A,A.A.A,asset,missing,1000,1000
1,3.761165e+05,5.575322,3,1,0,0,8,1,0,1.0,...,OPENING_BALANCE,USD,R2R,missing,A.D,A.D.A,asset,missing,1000,100400
2,1.366333e+06,6.135557,1,1,0,0,8,1,0,1.0,...,OPENING_BALANCE,USD,R2R,missing,A.H,A.H.A,asset,missing,1000,100760
3,2.282683e+06,6.358446,2,1,0,0,8,1,0,1.0,...,OPENING_BALANCE,USD,R2R,missing,A.B,A.B.A,asset,missing,1000,1100
4,6.021763e+05,5.779724,6,1,0,0,8,1,0,1.0,...,OPENING_BALANCE,USD,R2R,missing,A.C,A.C.A,asset,missing,1000,1200


In [6]:
# Logistic regression needs scaled numbers and one-hot encoded categories.
linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler()),
                ]
            ),
            numeric_features + boolean_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50, sparse_output=True),
            categorical_features + [account_feature],
        ),
    ],
    remainder="drop",
)

# A pipeline makes sure the same preprocessing is applied to train and test data.
logistic_model = Pipeline(
    steps=[
        ("prep", linear_preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                # Fraud rows are rare, so balanced weights give them more importance.
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_score = logistic_model.predict_proba(X_test)[:, 1]

print("Logistic precision:", round(precision_score(y_test, logistic_pred, zero_division=0), 4))
print("Logistic recall:", round(recall_score(y_test, logistic_pred, zero_division=0), 4))
print("Logistic F1:", round(f1_score(y_test, logistic_pred, zero_division=0), 4))


Logistic precision: 0.3941
Logistic recall: 0.6542
Logistic F1: 0.4919


In [7]:
# The tree model does not need scaling, but its categories still need numeric codes.
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features + boolean_features + [account_feature],
        ),
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
                encoded_missing_value=-1,
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)

n_numeric = len(numeric_features) + len(boolean_features) + 1
# These positions tell the gradient model which processed columns are categories.
categorical_positions = list(range(n_numeric, n_numeric + len(categorical_features)))

gradient_model = Pipeline(
    steps=[
        ("prep", tree_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=300,
                learning_rate=0.1,
                class_weight="balanced",
                categorical_features=categorical_positions,
                random_state=42,
            ),
        ),
    ]
)

# I train a second model to compare a linear method with a non-linear method.
gradient_model.fit(X_train, y_train)
gradient_pred = gradient_model.predict(X_test)
gradient_score = gradient_model.predict_proba(X_test)[:, 1]

print("Gradient precision:", round(precision_score(y_test, gradient_pred, zero_division=0), 4))
print("Gradient recall:", round(recall_score(y_test, gradient_pred, zero_division=0), 4))
print("Gradient F1:", round(f1_score(y_test, gradient_pred, zero_division=0), 4))


C:\Users\svgow\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\svgow\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\svgow\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\svgow\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\svgow\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Gradient precision: 0.723
Gradient recall: 0.6675
Gradient F1: 0.6942


In [8]:
# I keep all model metrics in the same format so the comparison is easier.
def score_method(method_name, y_true, y_pred, y_score):
    return {
        "method": method_name,
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "average_precision": round(average_precision_score(y_true, y_score), 4),
        "roc_auc": round(roc_auc_score(y_true, y_score), 4),
        "n_flagged": int(np.sum(y_pred)),
    }


# Recall by fraud type shows which kinds of fraud each model misses.
def recall_by_type(y_true, y_pred, fraud_type):
    result = pd.DataFrame(
        {
            "is_fraud": pd.Series(y_true).astype(bool),
            "caught": pd.Series(y_pred).astype(bool),
            "fraud_type": pd.Series(fraud_type),
        }
    )
    fraud_only = result[result["is_fraud"]]
    summary = fraud_only.groupby("fraud_type", observed=True)["caught"].agg(["size", "sum", "mean"])
    summary.columns = ["fraud_rows", "caught", "recall"]
    return summary.sort_values("fraud_rows", ascending=False).reset_index()

results_table = pd.DataFrame(
    [
        score_method("Logistic regression", y_test, logistic_pred, logistic_score),
        score_method("Gradient boosting", y_test, gradient_pred, gradient_score),
    ]
)

logistic_by_type = recall_by_type(y_test, logistic_pred, eval_fraud_type)
logistic_by_type.insert(0, "method", "Logistic regression")

gradient_by_type = recall_by_type(y_test, gradient_pred, eval_fraud_type)
gradient_by_type.insert(0, "method", "Gradient boosting")

recall_table = pd.concat([logistic_by_type, gradient_by_type], ignore_index=True)

results_table.to_csv(OUT_DIR / "supervised_results.csv", index=False)
recall_table.to_csv(OUT_DIR / "supervised_recall_by_fraud_type.csv", index=False)

results_table


,method,precision,recall,f1,average_precision,roc_auc,n_flagged
0,Logistic regression,0.3941,0.6542,0.4919,0.6724,0.8425,13491
1,Gradient boosting,0.7230,0.6675,0.6942,0.7176,0.8472,7503
